# 11.3 UDP - Datagrams and Unreliability

**Prerequisites:** 11.1 Networking Fundamentals, 11.2 TCP Client and Server  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Sending without connecting - `sendto` and `recvfrom`
- ✅ **Message boundaries are preserved** - the opposite of 11.2
- 🔴 What "unreliable" costs you: loss, reordering, duplication
- 🔴 **Truncation** - and why it behaves differently on Windows and Unix
- Datagram size limits, and the number you should actually use
- `connect()` on a UDP socket, which does not connect anything
- Building request/reply with timeouts and retries - reliability by hand
- When UDP is the right answer, and when it is not

---

## No connection, no handshake

UDP throws away almost everything TCP does. There is no `listen()`, no `accept()`, no `connect()` required — you address every message individually, like posting a postcard rather than making a phone call.

```
   TCP                              UDP
   ────────────────────────         ──────────────────────────
   socket(SOCK_STREAM)              socket(SOCK_DGRAM)
   bind() / listen() / accept()     bind()
   conn.recv(n)                     sock.recvfrom(n) -> (data, sender)
   conn.sendall(data)               sock.sendto(data, (host, port))
   ────────────────────────         ──────────────────────────
   one socket per client            ONE socket serves everybody
```

`recvfrom()` returns **`(data, address)`** — since there is no connection, the sender's address arrives with each message. That is how a UDP server knows where to reply.

> **What you are trading.** TCP gives you delivery, ordering, deduplication and flow control. UDP gives you none of them, and in exchange it gives you no handshake latency, no connection state per client, and no head-of-line blocking.

In [ ]:
import socket
import threading
import time

# ---- a server is just a bound socket ----
server = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
server.bind(("127.0.0.1", 0))
server.settimeout(2.0)                 # 🔴 as always
server_address = server.getsockname()
print("server bound to", server_address)

# ---- and a client is just a socket ----
client = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
client.settimeout(2.0)

client.sendto(b"metric cpu=0.82", server_address)

data, sender = server.recvfrom(4096)
print("server received:", data)
print("        from    :", sender, " <- the client's ephemeral port")

# reply to whoever sent it
server.sendto(b"ack", sender)
reply, _ = client.recvfrom(4096)
print("client received :", reply)

print("\nNo listen(), no accept(), no connection. Two sockets, four calls.")

## ✅ Boundaries survive - the exact opposite of TCP

In **11.2**, three `sendall(b"AAA")`/`(b"BBB")`/`(b"CCC")` calls arrived as one `b"AAABBBCCC"`. Over UDP, **one `sendto` is one `recvfrom`** — always. A datagram is an indivisible unit: it arrives whole, or it does not arrive.

This is genuinely useful. If your data is naturally message-shaped — a metric reading, a position update, a DNS query — UDP hands you framing for free, and all the framing code from 11.2 disappears.

The catch is in the next section.

In [ ]:
for message in (b"AAA", b"BBB", b"CCC"):
    client.sendto(message, server_address)

time.sleep(0.2)

received = []
for _ in range(3):
    data, _ = server.recvfrom(4096)
    received.append(data)

print("three sendto() calls ->", received)
print()
print("Compare 11.2, where the identical pattern over TCP produced")
print("b'AAABBBCCC' from a single recv(). UDP keeps every message intact.")

## 🔴 Truncation - the datagram that arrives half-eaten

Because a datagram is atomic, **your receive buffer must be big enough for the whole thing**. If it is not, the excess is gone — not queued, not returned by a second call. Discarded.

This is where the platforms disagree, and it is worth knowing which you are on:

| Platform | `recvfrom(100)` on a 2000-byte datagram |
|---|---|
| **Linux / macOS** | returns 100 bytes, **silently discards 1900** |
| **Windows** | raises `OSError` **WinError 10040** |

The Unix behaviour is the more dangerous of the two: no error, and 95% of your message quietly vanished. The next cell demonstrates whichever your machine does, and then confirms the remainder really is unrecoverable.

In [ ]:
big = b"X" * 2000
client.sendto(big, server_address)
time.sleep(0.1)

print(f"sent {len(big)} bytes, receiving with a 100-byte buffer:")
try:
    data, _ = server.recvfrom(100)
    print(f"  got {len(data)} bytes and NO error")
    print("  ^ Unix-style silent truncation - the other 1900 bytes are gone")
except OSError as exc:
    code = getattr(exc, "winerror", None) or exc.errno
    print(f"  OSError ({code}): message larger than the buffer")
    print("  ^ Windows-style - it refuses rather than truncating")

print("\nis the remainder still queued for a second read?")
try:
    leftover, _ = server.recvfrom(4096)
    print(f"  a second recvfrom got {len(leftover)} bytes")
except (TimeoutError, socket.timeout):
    print("  no - timed out. The rest of the datagram was discarded entirely.")

print("\n  Always size the buffer for your largest possible message.")

## How big can a datagram be?

| Limit | Value | Why |
|---|---|---|
| Theoretical maximum | **65507 bytes** | 65535 total − 20 IP header − 8 UDP header |
| Practical safe size | **~1472 bytes** | 1500 MTU − 20 − 8, so it fits one Ethernet frame |
| Guaranteed by IPv4 | 508 bytes | the minimum every network must accept without fragmenting |

Anything above the MTU gets **fragmented** across several IP packets — and if *any* fragment is lost, the whole datagram is discarded. So a 60 KB datagram is dramatically more likely to go missing than a 1 KB one.

**Keep datagrams under ~1400 bytes** unless you have measured otherwise. If your data is bigger, either split it yourself with sequence numbers, or use TCP.

In [ ]:
print("what actually sends on this machine:\n")
for size in (508, 1472, 8192, 65507, 65508):
    try:
        client.sendto(b"Y" * size, server_address)
        data, _ = server.recvfrom(70000)
        print(f"  {size:>6} bytes -> delivered {len(data)}")
    except OSError as exc:
        code = getattr(exc, "winerror", None) or exc.errno
        print(f"  {size:>6} bytes -> refused (error {code}): too large for UDP")

print("\n  65507 is the hard ceiling: 65535 - 20 (IP) - 8 (UDP).")
print("  But 'it sent' is not 'it is a good idea' - see the MTU note above.")

## 🔴 Unreliable means unreliable

UDP will not tell you a message was lost. There is no acknowledgement, no retry, no error — `sendto()` returns a byte count and reports success whether or not anything ever arrives.

Three things can happen to a datagram, and your code must tolerate all of them:

| | What you must do |
|---|---|
| **Loss** — never arrives | time out and retry, or accept the gap |
| **Reordering** — message 3 before message 2 | number your messages |
| **Duplication** — arrives twice | make handling idempotent, or discard by number |

On loopback none of these happen, which is exactly why UDP bugs survive testing. The cell below **simulates a lossy, reordering network** so you can see what your code has to cope with.

In [ ]:
import random


def lossy_send(sock, payload, address, *, loss=0.3, duplicate=0.25, rng=None):
    """Send over a deliberately awful network. Returns what it did."""
    rng = rng or random
    if rng.random() < loss:
        return "dropped"                 # never sent; sender cannot tell
    sock.sendto(payload, address)
    if rng.random() < duplicate:
        sock.sendto(payload, address)
        return "sent twice"
    return "sent"


# Seeded so the cell is reproducible - and chosen deliberately, because the
# first seed tried dropped nothing at all and demonstrated the opposite of
# the point. A 30% loss rate really can produce ten clean deliveries.
rng = random.Random(10)
outcomes = []
for n in range(1, 11):
    outcomes.append((n, lossy_send(client, f"msg-{n}".encode(), server_address, rng=rng)))

time.sleep(0.2)

arrived = []
while True:
    try:
        data, _ = server.recvfrom(4096)
        arrived.append(data.decode())
    except (TimeoutError, socket.timeout):
        break

print("what the sender did:")
for n, what in outcomes:
    print(f"   msg-{n:<3} {what}")

print(f"\nsent 10 messages, {len(arrived)} datagrams arrived:")
print("  ", arrived)

missing = [f"msg-{n}" for n in range(1, 11) if f"msg-{n}" not in arrived]
dupes = sorted({m for m in arrived if arrived.count(m) > 1})
print("\n  lost      :", missing or "none")
print("  duplicated:", dupes or "none")
print("\n🔴 sendto() reported success for every one of these. UDP will never")
print("   tell you a message was lost - that is the whole deal.")

## `connect()` on a UDP socket

Confusingly, UDP sockets *have* a `connect()`. It does **not** connect to anything and sends no packets. It just records a default peer, which gets you three things:

- `send()` and `recv()` work, so you can drop the `to`/`from` suffixes
- datagrams from any *other* address are filtered out
- 🔴 you start receiving **ICMP errors** reliably — so `ConnectionRefusedError` becomes possible when nothing is listening

That last point is the useful one: it is the only feedback UDP ever gives you.

> ⚠️ **Platform note.** On Windows an *unconnected* UDP socket can also be hit by the ICMP response — it surfaces as `ConnectionResetError` (WinError 10054) on the **next** `recvfrom()`, which is confusing because that call has nothing to do with the failed send. On Linux an unconnected socket ignores ICMP entirely. The cell below handles both, and this is a good reason to `connect()` UDP client sockets: the error then arrives predictably.

In [ ]:
probe = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
probe.settimeout(1.0)

# A port with nothing on it: bind one, note the number, close it.
spare = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
spare.bind(("127.0.0.1", 0))
dead_address = spare.getsockname()
spare.close()

print("-- unconnected: sending into the void --")
sent = probe.sendto(b"anyone there?", dead_address)
print(f"  sendto() returned {sent} - reported success")
try:
    probe.recvfrom(4096)
    print("  something came back (unexpected)")
except ConnectionResetError:
    print("  ConnectionResetError on the next recvfrom()")
    print("  ^ Windows: the ICMP 'port unreachable' surfaced here, on a call")
    print("    that had nothing to do with the send that caused it")
except (TimeoutError, socket.timeout):
    print("  recvfrom timed out - no error, no clue anything was wrong")
    print("  ^ Linux: an unconnected socket ignores the ICMP reply entirely")

print("\n-- connected: the same thing, with predictable feedback --")
bound = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
bound.settimeout(1.0)
bound.connect(dead_address)          # records a default peer; sends nothing
try:
    bound.send(b"anyone there?")     # note: send(), not sendto()
    time.sleep(0.2)
    bound.recv(4096)
    print("  nothing came back - ICMP was filtered or suppressed")
except (ConnectionRefusedError, ConnectionResetError) as exc:
    print(f"  {type(exc).__name__} - an ICMP 'port unreachable' came back")
    print("  ^ this is the feedback an unconnected socket may never give you")
except (TimeoutError, socket.timeout):
    print("  timed out - ICMP is often blocked entirely, so this is common too")
finally:
    bound.close()
    probe.close()

## Building reliability yourself

If you need UDP's speed *and* delivery, you implement the missing parts. The minimum viable version is **sequence number + acknowledgement + timeout + retry**:

```
   sender                          receiver
   ──────                          ────────
   send(seq=5, data)   ────────>   process it
   wait for ack, 0.2s   <────────  send(ack=5)
        │
        └─ no ack? send it again, up to N times
```

> **Be honest about what this is.** Add retries, ordering, deduplication, flow control and congestion control, and you have rewritten TCP — worse, and with bugs. Do this only for a narrow, well-understood case. QUIC (the basis of HTTP/3) is what it looks like done properly, and it took years.

The example below sends over the lossy channel from earlier and retries until acknowledged.

In [ ]:
def reliable_send(sock, payload, address, *, seq, retries=5, timeout=0.25, rng=None):
    """sendto + wait for ack, retrying on loss. Returns attempts used, or None."""
    framed = f"{seq}|".encode() + payload
    original = sock.gettimeout()
    sock.settimeout(timeout)
    try:
        for attempt in range(1, retries + 1):
            lossy_send(sock, framed, address, loss=0.5, duplicate=0.0, rng=rng)
            try:
                ack, _ = sock.recvfrom(64)
                if ack == f"ack:{seq}".encode():
                    return attempt
            except (TimeoutError, socket.timeout):
                continue                 # lost - go round again
        return None
    finally:
        sock.settimeout(original)


def ack_server(sock, stop_flag, seen, rng):
    while not stop_flag.is_set():
        try:
            data, sender = sock.recvfrom(4096)
        except (TimeoutError, socket.timeout):
            continue
        seq = data.split(b"|", 1)[0].decode()
        seen.append(seq)                 # the application has now processed it
        # 🔴 The ack can be lost too - and THAT is what creates duplicates.
        lossy_send(sock, f"ack:{seq}".encode(), sender,
                   loss=0.4, duplicate=0.0, rng=rng)


stop_flag = threading.Event()
seen = []
worker = threading.Thread(
    target=ack_server, args=(server, stop_flag, seen, random.Random(99)), daemon=True)
worker.start()

rng = random.Random(1)
for seq in range(1, 6):
    attempts = reliable_send(client, b"deploy", server_address, seq=seq, rng=rng)
    if attempts:
        print(f"  seq {seq}: sender saw an ack after {attempts} attempt(s)")
    else:
        print(f"  seq {seq}: GAVE UP after 5 attempts")

stop_flag.set()
worker.join(timeout=3)

print("\nsequences the RECEIVER actually processed:", seen)
duplicated = sorted({s for s in seen if seen.count(s) > 1})
print("processed more than once            :", duplicated or "none")

if duplicated:
    print("\n🔴 There it is. The receiver processed", duplicated, "twice, because")
    print("   its ack was lost and the sender retried something already done.")
    print("   The sender cannot tell this apart from the message being lost.")
else:
    print("\n(No duplicate this run - re-run with a different seed to see one.)")
print("\nThis is why 'at least once' delivery needs idempotent handling:")
print("dedupe on the sequence number, or make reprocessing harmless.")

## When to use UDP

| Use UDP when | Because |
|---|---|
| Late data is worthless | Live audio/video: a late frame is worse than a dropped one |
| One small request, one small reply | DNS: a handshake would cost more than the query |
| Broadcast or multicast to many | TCP cannot do one-to-many at all |
| Very high message rates, loss tolerable | metrics, `syslog`, game state |

| Use TCP when | Because |
|---|---|
| Every byte must arrive | file transfer, databases, APIs |
| Data is bigger than one datagram | TCP handles the splitting |
| Order matters | TCP handles it; you would have to |
| You are not sure | it is the right default |

> **The test.** If you find yourself adding retries, sequence numbers and acknowledgements to UDP, ask what you are gaining over TCP. Sometimes the answer is real — head-of-line blocking, or one-to-many. Often it is that UDP felt faster.

In [ ]:
# ---- tidy up ----
server.close()
client.close()

alive = [t.name for t in threading.enumerate() if t is not threading.main_thread()]
print("sockets closed")
print("background threads still alive:", alive or "none")

---

## Common Mistakes & Pitfalls

1. 🔴 **Assuming a datagram arrived because `sendto()` succeeded.** It reports that the bytes left your machine, nothing more.
2. 🔴 **Using a receive buffer smaller than the largest datagram.** Unix truncates silently; Windows raises. Either way the remainder is destroyed.
3. **Sending datagrams larger than the MTU.** They fragment, and losing any one fragment loses the whole message.
4. **Assuming order.** Datagram 3 can overtake datagram 2. Number them if it matters.
5. **Forgetting that retries create duplicates.** A lost *ack* makes the sender resend something already processed - handle it idempotently.
6. **Testing only on loopback.** Loopback never loses, reorders or duplicates, so every UDP bug survives your test suite.
7. **Expecting an error when nothing is listening.** Only a `connect()`-ed UDP socket surfaces ICMP errors, and even then they are often filtered.
8. **Reimplementing TCP on top of UDP.** If you need all of TCP's guarantees, use TCP.

## Best Practices

- Size receive buffers for the largest message you will ever send.
- Keep datagrams under ~1400 bytes to avoid fragmentation.
- Put a sequence number in every message if order or deduplication matters.
- Set a timeout and a retry limit on any request/reply exchange.
- Make message handling idempotent, so a duplicate is harmless.
- Test against a simulated lossy channel, as this notebook does - not just loopback.
- Use `connect()` on a UDP client socket to filter stray senders and surface ICMP errors.
- Default to TCP; choose UDP deliberately, with a reason you can state.

## Practice Exercises

Try these before moving on.

1. Change the truncation cell to receive with a 4096-byte buffer. Does the 2000-byte datagram now arrive intact on your platform?
2. Add reordering to `lossy_send` by queueing datagrams and flushing them in shuffled order. Then write a receiver that restores the correct order using sequence numbers.
3. Make `ack_server` ignore a sequence number it has already processed, and prove that duplicates no longer reach the application.
4. Measure the real fragmentation threshold on your machine: send increasing sizes to a remote host and find where delivery becomes unreliable.
5. Write a UDP time server: the client sends a request, the server replies with the current time, and the client retries up to three times before giving up.
6. 🔴 Take the reliable-send example and add exponential backoff between retries (0.25s, 0.5s, 1s). Why is a fixed short retry interval dangerous on a congested network?
7. Compare: transfer 1 MB over TCP, then over UDP in 1400-byte datagrams with acknowledgements. Time both, and count how much code each needed.